In [0]:
import requests # this is used to make api http/api calls 
import json # used to handles json data 
from pyspark.sql.functions import * # imports Spark transformation functions
from pyspark.sql.types import * # defines Spark schemas/datatypes

In [0]:
%sql 
CREATE CATALOG IF NOT EXISTS cricket_catalog; -- Top-level project container.

CREATE SCHEMA IF NOT EXISTS cricket_catalog.cricket_schema; -- Database section inside catalog.

CREATE VOLUME IF NOT EXISTS cricket_catalog.cricket_schema.raw_volume; -- Storage area for raw files.

In [0]:
# # this is other approach 
# API_KEY = '06d6d1ef-7d88-43ed-b188-b9f099db5c96'  
# # API key used for authenticating with CricAPI

# url = f"https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}&offset=0"
# # API endpoint URL for fetching current cricket matches

# response = requests.get(url)
# # Sends an HTTP GET request to the CricAPI server

# data = response.json()
# # Converts JSON response into a Python dictionary object

# print(json.dumps(data, indent=2))
# # Pretty prints the JSON response with indentation for readability

In [0]:
# import requests
# import json

# API_KEY = "06d6d1ef-7d88-43ed-b188-b9f099db5c966"
# # API key used for authentication with CricAPI

# url = f"https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}&offset=0"
# # API endpoint for fetching current cricket matches

# try:

#     response = requests.get(url, timeout=30)
#     # Sends HTTP GET request to API server

#     response.raise_for_status()
#     # Raises exception automatically if HTTP request failed

#     print("Status Code:", response.status_code)
#     # Prints response status code

#     data = response.json()
#     # Converts JSON response into Python dictionary

#     print(json.dumps(data, indent=4))
#     # Pretty prints JSON response for readability

# except requests.exceptions.HTTPError as http_err:
#     print("HTTP Error:", http_err)
#     # Handles HTTP-related errors like 404, 401, 500

# except requests.exceptions.Timeout:
#     print("Request timed out")
#     # Handles request timeout errors
    
# except requests.exceptions.RequestException as req_err:
#     print("Request Error:", req_err)
#     # Handles general request-related errors

# except Exception as e:
#     print("Unexpected Error:", e)
#     # Handles any other unexpected errors

In [0]:
# raw_file_path=f'{base_path}/current_matches_raw.json'
# with open (raw_file_path, 'w') as file:
#     json.dump(api_data,file)

#     print("RAW API data is save at the: ",raw_file_path)

In [0]:
import requests
import json

API_KEY = "06d6d1ef-7d88-43ed-b188-b9f099db5c966"
url = f"https://api.cricapi.com/v1/matches?apikey={API_KEY}&offset=0"

response = requests.get(url)
data = response.json()

# Save raw JSON to Volume
file_path = "/Volumes/cricket_catalog/cricket_schema/raw_volume/matches_raw.json"
with open(file_path, "w") as f:
    json.dump(data, f)

print("Raw data saved!")

In [0]:
from pyspark.sql.functions import *

# Read the JSON file
df_bronze = spark.read.option("multiLine", "true").json(file_path)

# Write as Delta table
df_bronze.write.format("delta").mode("overwrite").saveAsTable("cricket_catalog.cricket_schema.bronze_matches")
print("Bronze table created!")